# Module 4 | Class 3 Assignment: Decision Tree vs Random Forest Comparison

**Objective:** Train both a Decision Tree and a Random Forest on the Telco Churn dataset and compare their performance to understand why ensembles tend to outperform single models.

**Dataset:** Telco Customer Churn — [Kaggle Link](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)


## Setup: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("All libraries imported successfully!")


## Task 1: Load and Split Data

In [ ]:
# Step 1: Load the Telco Customer Churn dataset
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Dataset loaded! Shape: {df.shape}")
print()
print("First 3 rows:")
df.head(3)


In [ ]:
# ── Preprocessing (since we may not have Module 3 output) ──────────────────

# Drop customerID (not a useful feature)
df.drop(columns=['customerID'], inplace=True)

# TotalCharges has some spaces — convert to numeric and fill NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Encode binary/categorical columns
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

# One-hot encode remaining categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"Preprocessed shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


In [ ]:
# Step 2: Define features (X) and target (y), then split
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]} rows")
print(f"Test set     : {X_test.shape[0]} rows")
print(f"Churn rate (train): {y_train.mean():.2%}")
print(f"Churn rate (test) : {y_test.mean():.2%}")


## Task 2: Train a Decision Tree

In [ ]:
# Step 1: Train Decision Tree with default parameters
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree trained successfully!")


In [ ]:
# Step 2: Evaluate Decision Tree
dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_f1       = f1_score(y_test, y_pred_dt)

print("=== Decision Tree Performance ===")
print(f"Accuracy : {dt_accuracy:.4f}")
print(f"F1 Score : {dt_f1:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_dt, target_names=['No Churn', 'Churn']))


## Task 3: Train a Random Forest

In [ ]:
# Step 1: Train Random Forest with 100 trees
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest trained successfully!")


In [ ]:
# Step 2: Evaluate Random Forest
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1       = f1_score(y_test, y_pred_rf)

print("=== Random Forest Performance ===")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))


## Task 4: Visualize the Decision Tree

In [ ]:
# Plot Decision Tree (limited to depth=3 for readability)
plt.figure(figsize=(20, 10))
plot_tree(
    dt,
    max_depth=3,
    filled=True,
    feature_names=X.columns.tolist(),
    class_names=['No Churn', 'Churn'],
    fontsize=9,
    rounded=True,
    impurity=True
)
plt.title('Decision Tree (max_depth=3 shown)', fontsize=16, pad=20)
plt.tight_layout()
plt.show()


### Reading the Tree

Each node in the tree shows:
- **Splitting condition** — e.g., `Contract_Two year <= 0.5` (True goes left, False goes right)
- **Gini impurity** — how mixed the classes are at that node (0 = pure, 0.5 = maximally mixed)
- **Samples** — how many training examples reach this node
- **Value** — `[No Churn count, Churn count]` at that node
- **Class** — the majority class prediction if we stop here

**Example decision path:** Starting at the root, if a customer has no two-year contract AND has high monthly charges AND has been a customer for a short time → the tree predicts **Churn**. This aligns with business intuition: new customers on flexible plans paying high prices are most likely to leave.


## Task 5: Extract and Plot Feature Importances

In [ ]:
# Step 1: Extract feature importances from Random Forest
importances = pd.Series(rf.feature_importances_, index=X.columns)
top_15 = importances.sort_values(ascending=True).tail(15)

print("Top 15 Most Important Features:")
print(top_15.sort_values(ascending=False).to_string())


In [ ]:
# Step 2: Plot as horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 8))
top_15.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')

ax.set_title('Top 15 Feature Importances (Random Forest)', fontsize=14, pad=15)
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.grid(axis='x', linestyle='--', alpha=0.5)

for bar in ax.patches:
    ax.text(
        bar.get_width() + 0.001,
        bar.get_y() + bar.get_height() / 2,
        f'{bar.get_width():.4f}',
        va='center', ha='left', fontsize=9
    )

plt.tight_layout()
plt.show()


### Feature Importance Note

Feature importance in Random Forests is measured by how much each feature reduces impurity (Gini) on average across all 100 trees. **Higher score = the model relies on that feature more for splitting.**

> ⚠️ **Important caveat:** A high importance score does not mean the feature *causes* churn. For example, `tenure` may rank highly because long-term customers are less likely to churn — but that does not mean extending a contract artificially will reduce churn. Correlation ≠ causation.


## Task 6: Model Comparison

In [ ]:
# Summary comparison table
comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Accuracy': [dt_accuracy, rf_accuracy],
    'F1 Score': [dt_f1, rf_f1]
}).set_index('Model')

comparison = comparison.round(4)
print("=== Model Comparison Table ===")
print(comparison.to_string())
comparison


In [ ]:
# Visual comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

models = ['Decision Tree', 'Random Forest']
colors = ['#e07b54', '#4c8be0']

for ax, metric, values in zip(
    axes,
    ['Accuracy', 'F1 Score'],
    [[dt_accuracy, rf_accuracy], [dt_f1, rf_f1]]
):
    bars = ax.bar(models, values, color=colors, width=0.5, edgecolor='white')
    ax.set_ylim(0, 1)
    ax.set_title(metric, fontsize=13)
    ax.set_ylabel('Score', fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{val:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold'
        )

plt.suptitle('Decision Tree vs Random Forest', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### Written Comparison

**Which model performed better?**  
The Random Forest outperforms the Decision Tree on both Accuracy and F1 Score. This is the expected result in virtually all real-world classification problems.

**Why does Random Forest usually win?**  
A single Decision Tree with default parameters grows until every leaf is pure — it memorizes the training data (overfitting) and fails to generalize to unseen examples. Random Forest counters this with two mechanisms: (1) **Bagging** — each of the 100 trees is trained on a random bootstrap sample of the training data, so no single noisy example dominates; (2) **Feature randomness** — at each split, only a random subset of features is considered, which forces the trees to be diverse. When 100 diverse trees vote on a prediction, errors made by individual trees tend to cancel out, yielding a far more robust final prediction.

**What is the tradeoff?**  
The main cost of Random Forest is **interpretability vs speed**. A single Decision Tree can be plotted and fully understood in seconds — a manager can trace a single customer's path from root to leaf and understand exactly why the model predicts churn. With 100 trees, that transparency disappears entirely. Random Forest also takes roughly 100× longer to train and predict, and requires more memory. For a problem where stakeholders need to explain individual decisions (e.g., regulatory compliance), a Decision Tree with tuned depth (`max_depth=5`) may be preferred even at a small accuracy cost.


In [ ]:
# Final summary printout
print("=" * 50)
print("        ASSIGNMENT COMPLETE — FINAL SUMMARY")
print("=" * 50)
print(f"{'Model':<20} {'Accuracy':>10} {'F1 Score':>10}")
print("-" * 50)
print(f"{'Decision Tree':<20} {dt_accuracy:>10.4f} {dt_f1:>10.4f}")
print(f"{'Random Forest':<20} {rf_accuracy:>10.4f} {rf_f1:>10.4f}")
print("=" * 50)
winner = "Random Forest" if rf_f1 > dt_f1 else "Decision Tree"
print(f"Best model by F1: {winner}")
